# PROVISIONAL — CO2 y temperatura de los backups (LAB019-021)

**Notebook descartable** (se eliminará): solo grafica el contenido de los backups globales
descargados de la RPi5, acotado a la campaña LAB019-021. **No calcula nada del modelo, no guarda
figuras en `results/`** — todas las figuras quedan inline en los outputs.

- Fuentes: `data/mem2026/LAB019-021/backup_global_co2.csv` y `backup_global_temperatura.csv`
  (backup global desde el 11/05; aquí se filtra a la ventana 21–28/09/2026).
- Mapeo: F1 = LAB019 (A), F2 = LAB020 (C), F3 = LAB021 (A). t0 = 21/09/2026 **17:00** (aplicación de levadura; el pulso 1 de nutrición se dosificó al mosto ANTES del inóculo, por eso la muestra 01 es pre-inoculación).
- La nutrición de esta campaña fue **manual** (no bombas): `nutricion_activa` del log NO registra
  los pulsos; se dibujan como líneas a partir de la hoja `Nutricion` del Excel offline
  (F3 23/09 16:30 · F1 24/09 09:20 · F2 24/09 16:40; pulso 1 = misma dosis, al mosto antes del inóculo).

In [ ]:
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPLICA_DIR = Path.cwd()
for p in [Path.cwd(), *Path.cwd().resolve().parents]:
    if (p / "fermentation_model").exists():
        REPLICA_DIR = p / "fermentation_model" / "data" / "mem2026" / "LAB019-021"
        break
CO2_PATH = REPLICA_DIR / "backup_global_co2.csv"
TEMP_PATH = REPLICA_DIR / "backup_global_temperatura.csv"
assert CO2_PATH.exists() and TEMP_PATH.exists()

T0 = pd.Timestamp(2026, 9, 21, 17, 0)  # aplicacion de levadura (inoculacion manual), confirmado por laboratorio
WIN_START, WIN_END = pd.Timestamp(2026, 9, 21, 0, 0), pd.Timestamp(2026, 9, 28, 12, 0)
FERM = {"F1": "LAB019 (F1, A)", "F2": "LAB020 (F2, C)", "F3": "LAB021 (F3, A)"}
PULSES_H = {"F1": (datetime(2026, 9, 24, 9, 20) - T0.to_pydatetime()).total_seconds() / 3600,
            "F2": (datetime(2026, 9, 24, 16, 40) - T0.to_pydatetime()).total_seconds() / 3600,
            "F3": (datetime(2026, 9, 23, 16, 30) - T0.to_pydatetime()).total_seconds() / 3600}
SP_EVENTS = {"F1": [(99, 21), (171, 16)], "F2": [(123, 18), (219, 21)], "F3": [(96, 21), (168, 16)]}
COLORS = {"F1": "#4c72b0", "F2": "#55a868", "F3": "#c44e52"}

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9.5, "axes.titlesize": 10.5})
print("paths OK:", CO2_PATH.name, TEMP_PATH.name)

In [ ]:
co2 = pd.read_csv(CO2_PATH, usecols=["timestamp", "fermentador", "flow_sccm", "flow_filt_sccm", "status"], dtype=str)
co2["timestamp"] = pd.to_datetime(co2["timestamp"], errors="coerce")
bad_co2 = int(co2["timestamp"].isna().sum())
for c in ("flow_sccm", "flow_filt_sccm"):
    co2[c] = pd.to_numeric(co2[c], errors="coerce")
co2["status"] = co2["status"].astype("category")
co2["fermentador"] = co2["fermentador"].astype("category")
co2 = co2.dropna(subset=["timestamp"])
co2 = co2[(co2["timestamp"] >= WIN_START) & (co2["timestamp"] <= WIN_END)].copy()
print(f"CO2: {bad_co2} timestamps ilegibles en todo el archivo; en ventana: {len(co2)} filas")
summary = co2.groupby("fermentador", observed=True).agg(
    filas=("flow_sccm", "size"), t_ini=("timestamp", "min"), t_fin=("timestamp", "max"),
    raw_max=("flow_sccm", "max"), filt_max=("flow_filt_sccm", "max"))
display(summary)
display(co2.groupby(["fermentador", "status"], observed=True).size().rename("filas").reset_index())

In [ ]:
temp = pd.read_csv(TEMP_PATH, usecols=["timestamp", "fermentador", "T", "SP"], dtype=str)
temp["timestamp"] = pd.to_datetime(temp["timestamp"], errors="coerce")
bad_temp = int(temp["timestamp"].isna().sum())
for c in ("T", "SP"):
    temp[c] = pd.to_numeric(temp[c], errors="coerce")
temp["fermentador"] = temp["fermentador"].astype("category")
temp = temp.dropna(subset=["timestamp"])
temp = temp[(temp["timestamp"] >= WIN_START) & (temp["timestamp"] <= WIN_END)].copy()
print(f"Temperatura: {bad_temp} timestamps ilegibles en todo el archivo "
      f"(linea 3849209 del backup: dos registros fusionados, 17/08); en ventana: {len(temp)} filas")
display(temp.groupby("fermentador", observed=True).agg(
    filas=("T", "size"), T_min=("T", "min"), T_max=("T", "max"), T_med=("T", "median"),
    SP_unicos=("SP", lambda s: ", ".join(f"{v:g}" for v in sorted(s.unique())))))

## CO2: raw vs filtrado, por fermentador

Línea fina translúcida = `flow_sccm` (raw, diezmada 1 de cada 30 registros ≈ 30 s);
línea sólida = `flow_filt_sccm` (mediana 60 s). Rayado vertical = pulso manual de nutrición;
punteado = cambios de setpoint.

In [ ]:
def hours_since_t0(index):
    return (pd.DatetimeIndex(index) - T0).total_seconds() / 3600.0

co2_by = {f: co2[co2["fermentador"] == f].set_index("timestamp").sort_index() for f in FERM}
temp_by = {f: temp[temp["fermentador"] == f].set_index("timestamp").sort_index() for f in FERM}

fig, axes = plt.subplots(3, 1, figsize=(13, 8.2), sharex=True)
for ax, (f, label) in zip(axes, FERM.items()):
    sub = co2_by[f]
    raw = sub["flow_sccm"].iloc[::30]
    filt = sub["flow_filt_sccm"].resample("60s").median()
    ax.plot(hours_since_t0(raw.index), raw.values, color=COLORS[f], lw=0.5, alpha=0.35, label="raw")
    ax.plot(hours_since_t0(filt.index), filt.values, color=COLORS[f], lw=1.4, label="filtrado (mediana 60 s)")
    ax.axvline(PULSES_H[f], color="k", ls="--", lw=1.0, alpha=0.6)
    ax.annotate("pulso N", (PULSES_H[f], ax.get_ylim()[1]), fontsize=8, ha="right", va="top", rotation=90)
    for t_sp, sp in SP_EVENTS[f]:
        ax.axvline(t_sp, color="gray", ls=":", lw=0.9, alpha=0.6)
    ax.set_ylabel(f"{label}\nCO2 (sccm)")
    ax.legend(loc="upper right", fontsize=8.5, frameon=True)
axes[-1].set_xlabel("t desde t0 = 21/09 17:00 (h)")
fig.suptitle("CO2 backups LAB019-021: raw vs filtrado (ventana 21–28/09/2026)", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.97))
display(fig)
plt.close(fig)

## Temperatura: medida vs setpoint

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 8.2), sharex=True)
for ax, (f, label) in zip(axes, FERM.items()):
    sub = temp_by[f]
    tt = sub["T"].resample("60s").median()
    sp = sub["SP"].resample("60s").median()
    ax.plot(hours_since_t0(tt.index), tt.values, color=COLORS[f], lw=1.1, label="T medida")
    ax.plot(hours_since_t0(sp.index), sp.values, color="k", lw=1.0, ls="--", alpha=0.7, label="SP")
    t_end_h = (sub.index[-1] - T0).total_seconds() / 3600.0
    for t_sp, spv in SP_EVENTS[f]:
        if t_sp <= t_end_h:
            ax.axvline(t_sp, color="gray", ls=":", lw=0.9, alpha=0.6)
    ax.set_ylabel(f"{label}\nT (°C)")
    ax.legend(loc="lower right", fontsize=8.5, frameon=True)
axes[-1].set_xlabel("t desde t0 = 21/09 17:00 (h)")
fig.suptitle("Temperatura backups LAB019-021: T medida vs setpoint", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.97))
display(fig)
plt.close(fig)

## Overlay comparativo (CO2 filtrado) y onset

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.6))
onset_rows = []
for f, label in FERM.items():
    filt = co2_by[f]["flow_filt_sccm"].resample("60s").median()
    h = hours_since_t0(filt.index)
    ax.plot(h, filt.values, color=COLORS[f], lw=1.3, label=label)
    above = np.flatnonzero(filt.fillna(0).values > 0.5)
    onset_rows.append({"fermentador": label,
                       "onset (>0.5 sccm)": f"{h[above[0]]:.1f} h" if len(above) else "no alcanzado",
                       "pico (sccm)": f"{np.nanmax(filt.values):.2f}",
                       "raw_max (sccm)": f"{co2_by[f]['flow_sccm'].max():.2f}"})
for f in FERM:
    ax.axvline(PULSES_H[f], color=COLORS[f], ls="--", lw=0.9, alpha=0.5)
ax.set_xlabel("t desde t0 = 21/09 17:00 (h)")
ax.set_ylabel("CO2 filtrado (sccm)")
ax.set_title("CO2 filtrado: comparación entre fermentadores (rayado = pulso N de cada uno)")
ax.legend(fontsize=9, frameon=True)
fig.tight_layout()
display(fig)
plt.close(fig)
display(pd.DataFrame(onset_rows))